# Regional VMD EOG removal — posterior-context

Matched ICA and GEVD-MWF posterior experts with raw and signed frontal support.

## Scientific question, inputs and contracts

Attach pinned corpus and passing research-readiness artifacts. CPU-only classical experiment independent of the GPU frontal sweep. Compare ICA and regularized GEVD-MWF with posterior-only, common-average, signed left/right/midline/common/right-minus-left, raw frontal and full-montage support; correct posterior outputs only. Metadata determines stable support layouts; missing-context flags are saved. Both experts fit only unscored calibration, including trial-join protection for ICA reference association. Pilot: one recipient per frozen fold, clean plus 0 dB blink/lateral/mixed; MWF ranks 1/2, both declared lag banks, thresholds .4/.6 and strengths .5/1. Full: all eligible recipients, -5/0/5 dB, ranks 1..4 and all declared thresholds/strengths. Save complete per-channel paper MSE/temporal RRMSE/Pearson/output-SNR/improvement, per-example shards, convergence failures and source-excluded recipe selection. Identity and fixed .01-ridge EOG regression controls share the original input. Calibration/scoring layouts and donor/recipient disjointness are enforced. Signed virtual channels are redundant; rank/regularization effects are disclosed. Partial preservation screening does not qualify a teacher; paired-correlation comparator and native OSF validation remain separate. Read docs/POSTERIOR_CONTEXT_PLAN.md.

## Fresh source and environment
Resolve latest published code once and pin it. New temporary clones avoid stale Python imports; original inputs remain immutable.

In [ ]:
import os, sys, json, tempfile, subprocess
from pathlib import Path
IN_KAGGLE = Path('/kaggle/input').is_dir()
if not IN_KAGGLE:
    raise RuntimeError('This numerical campaign runs on Kaggle only')
TEMP_ROOT = Path(tempfile.mkdtemp(prefix='vmd-eog-'))
CHECKOUT = TEMP_ROOT / 'repository'
REPOSITORY_URL = 'https://github.com/saltypal/Lateral-eye-Artifact-removal.git'
BRANCH = 'research/vmd-eog-removal-notebooks'
RUN_SPEC = {'stage': 'posterior-context', 'run_id': 'posterior-context-manual', 'git_sha': '', 'campaign_id': 'vmd-bandroute-20261010', 'profile': 'pilot', 'config_path': 'configs/bandroute.json', 'experiment': {'purpose': 'matched_posterior_context'}}
# Manual launches resolve latest once; CLI launches already have its exact SHA.
if not RUN_SPEC['git_sha']:
    line = subprocess.check_output(['git','ls-remote',REPOSITORY_URL,'refs/heads/'+BRANCH], text=True).strip()
    RUN_SPEC['git_sha'] = line.split()[0]
subprocess.check_call(['git','clone','--quiet','--branch',BRANCH,'--single-branch',REPOSITORY_URL,str(CHECKOUT)])
subprocess.check_call(['git','-C',str(CHECKOUT),'checkout','--quiet',RUN_SPEC['git_sha']])
ACTUAL_SHA = subprocess.check_output(['git','-C',str(CHECKOUT),'rev-parse','HEAD'],text=True).strip()
if ACTUAL_SHA != RUN_SPEC['git_sha']:
    raise RuntimeError('Source SHA mismatch')
ENV_ROOT = TEMP_ROOT / 'environment'
subprocess.check_call([sys.executable,'-m','venv','--without-pip','--system-site-packages',str(ENV_ROOT)])
ENV_PY = ENV_ROOT / 'bin/python'
subprocess.check_call([sys.executable,'-m','pip','--python',str(ENV_PY),'install','--quiet','--progress-bar','off','-r',str(CHECKOUT/'requirements.txt')])
PROCESS_ENV = os.environ.copy()
PROCESS_ENV['PYTHONPATH'] = str(CHECKOUT/'src')
PROCESS_ENV['PYTHONUNBUFFERED'] = '1'
PROCESS_ENV['OMP_NUM_THREADS'] = '2'
OUT = Path('/kaggle/working/results') / RUN_SPEC['campaign_id'] / RUN_SPEC['run_id']
OUT.mkdir(parents=True,exist_ok=False)
(OUT/'run_spec.json').write_text(json.dumps(RUN_SPEC,indent=2))
(OUT/'experiment.json').write_text(json.dumps(RUN_SPEC.get('experiment',{}),indent=2))
print('Exact Git SHA:', ACTUAL_SHA, 'Stage:', RUN_SPEC['stage'], 'Output:',OUT,flush=True)


## Inspect the pinned algorithm contract
The source modules remain inspectable in this checkout. Mode vectors, references and paired targets have distinct roles; the accompanying explanation defines their shapes and inference boundaries.

In [ ]:
CONFIGURATION = json.loads((CHECKOUT/RUN_SPEC.get('config_path','configs/campaign.json')).read_text())
print('Declared signal contract:', {key: CONFIGURATION[key] for key in ('fs','window','hop','seed')})
print('Preservation limits:', CONFIGURATION['preservation'])
print('Algorithm explanation: docs/SCIENTIFIC_EXPLANATION.md in the pinned checkout')
print('Modules: signal.py -> reference.py/frontal.py/posterior.py -> experiments.py -> review evidence')
print('Git source:', ACTUAL_SHA)


## Execute reusable modules
HEOG/VEOG are teacher/evaluation information. Clean targets cannot enter inference. Failed numerical or scientific gates stop dependent work.

In [ ]:
command = [str(ENV_PY),'-m','vmd_eog.runner','--stage',RUN_SPEC['stage'],
           '--input','/kaggle/input','--output',str(OUT),'--profile',RUN_SPEC['profile'],
           '--config',RUN_SPEC.get('config_path','configs/campaign.json'),
           '--experiment',str(OUT/'experiment.json')]
process = subprocess.run(command,cwd=CHECKOUT,env=PROCESS_ENV)
if process.returncode:
    raise RuntimeError('Stage failed; inspect saved execution_state.json and kernel logs')
print((OUT/'execution_state.json').read_text())


## Saved configurations and result tables
These are immutable outputs from this run. Readiness, teacher eligibility and final scientific qualification are distinct decisions.

In [ ]:
import csv
from itertools import islice
for name in ('vmd_search.csv','posterior_search.csv','review_methods.csv','condition_source_uncertainty.csv','paper_metric_fixture.csv','paper_paired_channels.csv','paper_native_channels.csv','paper_rest_bandpower_channels.csv','paper_paired_source_means.csv','paper_native_source_means.csv','paper_paired_permutation_tests.csv','paper_native_permutation_tests.csv','native_condition_channels.csv','native_condition_participants.csv','native_condition_permutations.csv','vmd_diagnosis_method_means.csv','vmd_diagnosis_clean_preservation.csv','vmd_mode_projection_diagnostics.csv','paper_plot_participant_conditions.csv','vmd_sobi_method_means.csv','vmd_sobi_clean_preservation.csv'):
    path = OUT/name
    if path.exists():
        print(name, '(first 8 saved rows; complete table remains in outputs)')
        with path.open() as handle:
            display(list(islice(csv.DictReader(handle),8)))
for name in ('posterior_context_search.csv','posterior_context_oof_scores.csv'):
    path = OUT/name
    if path.exists():
        print(name, '(first 8 rows; complete paper metrics and failure records saved)')
        with path.open() as handle:
            display(list(islice(csv.DictReader(handle),8)))
for name in ('selected_frontal.json','selected_posterior.json','classical_gate.json','paper_evaluation_protocol.json','paper_metric_fixture_summary.json'):
    if (OUT/name).exists():
        print(name, (OUT/name).read_text()[:8000])
review = OUT/'APPROACH_REVIEW.md'
if review.exists():
    from IPython.display import Markdown
    display(Markdown(review.read_text()))


## Inspect evidence and persist outputs
Native real EEG has no clean-reference SNR. Pass-through failures remain counted. Parameters and physiological frequency bands are distinct.

In [ ]:
from IPython.display import display, Image
for figure in sorted(OUT.glob('*.png')):
    display(Image(filename=str(figure)))
for report in sorted(OUT.glob('*summary.json')):
    print(report.name, report.read_text()[:16000])
print('Persist this completed kernel version and retrieve outputs before launching a dependent stage.')
